In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
np.random.seed(42)

num_samples = 1000

# Party A features (e.g., bank data)
XA = np.random.randn(num_samples, 5)

# Party B features (e.g., ecommerce data)
XB = np.random.randn(num_samples, 5)

# True weights for generating labels
true_w = np.random.randn(10, 1)

X_combined = np.concatenate([XA, XB], axis=1)
y = (X_combined @ true_w + np.random.randn(num_samples,1)*0.5 > 0).astype(int)

XA = torch.tensor(XA, dtype=torch.float32)
XB = torch.tensor(XB, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [3]:
class PartyA(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 4)

    def forward(self, x):
        return self.fc(x)

In [4]:
class PartyB(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 4)

    def forward(self, x):
        return self.fc(x)

In [5]:
class Aggregator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(8,1)

    def forward(self, a_out, b_out):
        combined = torch.cat([a_out, b_out], dim=1)
        return torch.sigmoid(self.fc(combined))

In [6]:
partyA = PartyA()
partyB = PartyB()
aggregator = Aggregator()

criterion = nn.BCELoss()

optimizerA = optim.Adam(partyA.parameters(), lr=0.01)
optimizerB = optim.Adam(partyB.parameters(), lr=0.01)
optimizerAgg = optim.Adam(aggregator.parameters(), lr=0.01)

In [7]:
epochs = 50

for epoch in range(epochs):

    optimizerA.zero_grad()
    optimizerB.zero_grad()
    optimizerAgg.zero_grad()

    # Each party computes local embeddings
    a_out = partyA(XA)
    b_out = partyB(XB)

    # Send embeddings to aggregator
    predictions = aggregator(a_out, b_out)

    # Compute loss
    loss = criterion(predictions, y)

    # Backpropagation
    loss.backward()

    optimizerA.step()
    optimizerB.step()
    optimizerAgg.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.6883
Epoch 10 | Loss: 0.5624
Epoch 20 | Loss: 0.4359
Epoch 30 | Loss: 0.3364
Epoch 40 | Loss: 0.2625


In [8]:
with torch.no_grad():

    a_out = partyA(XA)
    b_out = partyB(XB)

    preds = aggregator(a_out, b_out)
    preds = (preds > 0.5).float()

    accuracy = (preds == y).float().mean()

    print("\nFinal Accuracy:", accuracy.item())


Final Accuracy: 0.9229999780654907
